# Week 3.1: Tool Use - Building AI Agents

## Key Insight: Thought → Action → Observation → Answer

Your digital twin learns to USE TOOLS autonomously!

## What You'll Build
- Tools your AI can call (calculator, search, etc.)
- Agent that decides when to use tools
- Multi-step reasoning loops
- Digital twin with tool capabilities

In [1]:
import json
from typing import Dict, List, Any, Callable
from datetime import datetime

print('Week 3.1: Tool Use & Agents')
print('Goal: Make your digital twin DO things, not just talk!')

Week 3.1: Tool Use & Agents
Goal: Make your digital twin DO things, not just talk!


## Part 1: Creating Tools

A tool is a function your AI can call.

In [2]:
# Tool Example 1: Calculator
def calculator(expression: str) -> Dict[str, Any]:
    '''Calculate math expressions.'''
    try:
        result = eval(expression)  # Safe in controlled environment
        return {'result': result, 'expression': expression}
    except Exception as e:
        return {'error': str(e)}

# Tool Example 2: Get current time
def get_time() -> Dict[str, Any]:
    '''Get current time.'''
    now = datetime.now()
    return {
        'time': now.strftime('%H:%M:%S'),
        'date': now.strftime('%Y-%m-%d'),
        'day': now.strftime('%A')
    }

print('Calculator:', calculator('15 * 0.15'))
print('Time:', get_time())

Calculator: {'result': 2.25, 'expression': '15 * 0.15'}
Time: {'time': '19:56:44', 'date': '2026-10-29', 'day': 'Thursday'}


## Part 2: Tool Registry

Manage available tools.

In [3]:
class ToolRegistry:
    '''Manages available tools for the agent.'''

    def __init__(self):
        self.tools = {}

    def register(self, name: str, func: Callable, description: str):
        '''Register a tool.'''
        self.tools[name] = {
            'function': func,
            'description': description,
            'name': name
        }

    def get_tool(self, name: str) -> Callable:
        '''Get tool function by name.'''
        return self.tools[name]['function']

    def list_tools(self) -> str:
        '''Get formatted list of available tools.'''
        lines = ['Available tools:']
        for name, info in self.tools.items():
            lines.append(f"- {name}: {info['description']}")
        return '\n'.join(lines)

#Create registry
registry = ToolRegistry()
registry.register('calculator', calculator, 'Calculate math expressions')
registry.register('get_time', get_time, 'Get current date and time')

print(registry.list_tools())

Available tools:
- calculator: Calculate math expressions
- get_time: Get current date and time


## Part 3: ReAct Agent

Agent that can reason and act with tools.

**ReAct Loop:**
1. **Thought**: Decide what to do
2. **Action**: Use a tool
3. **Observation**: See result
4. **Answer**: Respond to user (or loop again)

In [4]:
class SimpleReActAgent:
    '''Simple agent that can use tools.'''

    def __init__(self, tool_registry: ToolRegistry):
        self.registry = tool_registry
        self.history = []

    def think(self, query: str) -> str:
        '''Decide what action to take.'''
        # Simplified: pattern matching for demo
        query_lower = query.lower()

        if any(word in query_lower for word in ['calculate', 'math', '+', '-', '*', '/']):
            return 'use_calculator'
        elif any(word in query_lower for word in ['time', 'date', 'day']):
            return 'use_time'
        else:
            return 'direct_answer'

    def act(self, action: str, query: str) -> Dict[str, Any]:
        '''Execute the action.'''
        if action == 'use_calculator':
            # Extract math expression from query
            import re
            match = re.search(r'[\d+\-*/().\s]+', query)
            if match:
                expr = match.group(0).strip()
                return self.registry.get_tool('calculator')(expr)
            return {'error': 'Could not extract expression'}

        elif action == 'use_time':
            return self.registry.get_tool('get_time')()

        else:
            return {'direct': query}

    def run(self, query: str) -> str:
        '''Run the agent on a query.'''
        # Step 1: Thought
        thought = self.think(query)
        self.history.append({'step': 'thought', 'content': thought})

        # Step 2: Action
        observation = self.act(thought, query)
        self.history.append({'step': 'action', 'content': observation})

        # Step 3: Answer
        if 'result' in observation:
            answer = f"The result is {observation['result']}"
        elif 'time' in observation:
            answer = f"It's {observation['time']} on {observation['date']} ({observation['day']})"
        elif 'error' in observation:
            answer = f"Error: {observation['error']}"
        else:
            answer = "I can help with calculations and time queries."

        return answer

# Test the agent
agent = SimpleReActAgent(registry)

print('\nAgent Tests:')
print('Q: What is 25 * 4?')
print('A:', agent.run('What is 25 * 4?'))
print('\nQ: What time is it?')
print('A:', agent.run('What time is it?'))


Agent Tests:
Q: What is 25 * 4?
A: Error: invalid syntax (<string>, line 0)

Q: What time is it?
A: It's 20:11:14 on 2026-10-29 (Thursday)


## Part 4: Integrating with Digital Twin

Add tools to your Week 1 digital twin.

In [5]:
class DigitalTwinWithTools:
    '''Digital twin that can use tools.'''

    def __init__(self, name: str):
        self.name = name
        self.registry = ToolRegistry()
        self.agent = None
        self.setup_tools()

    def setup_tools(self):
        '''Setup default tools.'''
        self.registry.register('calculator', calculator, 'Calculate math')
        self.registry.register('get_time', get_time, 'Get current time')
        self.agent = SimpleReActAgent(self.registry)

    def add_tool(self, name: str, func: Callable, description: str):
        '''Add a new tool.'''
        self.registry.register(name, func, description)

    def chat(self, message: str) -> str:
        '''Chat with tool use.'''
        return self.agent.run(message)

# Create digital twin with tools
twin = DigitalTwinWithTools('My Assistant')

print('\nDigital Twin with Tools:')
print(twin.chat('Calculate 47.50 * 0.15'))
print(twin.chat('What day is today?'))


Digital Twin with Tools:
The result is 7.125
It's 20:15:45 on 2026-10-29 (Thursday)


## Exercise 1: Create Custom Tools

**Task**: Add 2 custom tools to your digital twin.

Ideas:
- Memory search
- Note taker
- Task list
- Habit tracker

In [ ]:
# YOUR CODE HERE

def my_custom_tool(arg: str) -> Dict[str, Any]:
    '''Your tool description.'''
    # Implement your tool
    return {'result': 'your result'}

# Add to twin
# twin.add_tool('my_tool', my_custom_tool, 'Description')
# Test: twin.chat('query that uses your tool')

In [6]:
import re

# --- 1. Define In-Memory Databases for our Tools ---
notes_db = []
habits_db = {}

# --- 2. Define Custom Tool Functions ---
def note_taker(note_text: str) -> Dict[str, Any]:
    '''Saves a note to the database.'''
    note_text = note_text.strip()
    if not note_text:
        return {'error': 'Note text cannot be empty'}
    notes_db.append(note_text)
    return {'status': 'success', 'saved_note': note_text, 'all_notes': notes_db}

def habit_tracker(habit_name: str) -> Dict[str, Any]:
    '''Tracks and counts habit completions.'''
    habit_name = habit_name.strip().lower()
    if not habit_name:
        return {'error': 'Habit name cannot be empty'}
    habits_db[habit_name] = habits_db.get(habit_name, 0) + 1
    return {'status': 'success', 'habit': habit_name, 'count': habits_db[habit_name]}

# --- 3. Upgrade Agent to Support New Tools ---
class AdvancedReActAgent(SimpleReActAgent):
    '''An upgraded agent that recognizes note taking and habit tracking actions.'''

    def think(self, query: str) -> str:
        query_lower = query.lower()
        if any(word in query_lower for word in ['note', 'write down', 'remember']):
            return 'use_note_taker'
        elif any(word in query_lower for word in ['habit', 'track', 'log']):
            return 'use_habit_tracker'
        return super().think(query)

    def act(self, action: str, query: str) -> Dict[str, Any]:
        if action == 'use_note_taker':
            # Simple extraction: remove keywords to get the content
            content = re.sub(r'(?i)\b(write down|take note|note|remember)\b', '', query).strip()
            return self.registry.get_tool('note_taker')(content)

        elif action == 'use_habit_tracker':
            # Simple extraction: extract the habit name
            content = re.sub(r'(?i)\b(track|log|habit)\b', '', query).strip()
            return self.registry.get_tool('habit_tracker')(content)

        return super().act(action, query)

    def run(self, query: str) -> str:
        # Step 1: Thought
        thought = self.think(query)
        self.history.append({'step': 'thought', 'content': thought})

        # Step 2: Action
        observation = self.act(thought, query)
        self.history.append({'step': 'action', 'content': observation})

        # Step 3: Answer custom formatting for new tools
        if observation.get('status') == 'success' and 'saved_note' in observation:
            return f"Saved note: \"{observation['saved_note']}\". You now have {len(observation['all_notes'])} note(s) saved."
        elif observation.get('status') == 'success' and 'habit' in observation:
            return f"Logged habit: \"{observation['habit']}\". Total completions: {observation['count']}."

        return super().run(query)

In [7]:
# --- 4. Re-initialize DigitalTwin with the Advanced Agent ---
class AdvancedDigitalTwin(DigitalTwinWithTools):
    def setup_tools(self):
        super().setup_tools()
        # Override with our advanced agent
        self.agent = AdvancedReActAgent(self.registry)
        # Register custom tools
        self.add_tool('note_taker', note_taker, 'Save notes and reminders')
        self.add_tool('habit_tracker', habit_tracker, 'Track daily habit counts')

# Create digital twin
twin = AdvancedDigitalTwin('My Smart Assistant')

# --- 5. Test Custom Tools ---
print('Testing Note Taker:')
print('A:', twin.chat('Note buy groceries this afternoon'))
print('A:', twin.chat('Remember to call Sarah at 5pm'))

print('\nTesting Habit Tracker:')
print('A:', twin.chat('Track drink water'))
print('A:', twin.chat('Track exercise'))
print('A:', twin.chat('Track drink water'))

Testing Note Taker:
A: Saved note: "buy groceries this afternoon". You now have 1 note(s) saved.
A: Saved note: "to call Sarah at 5pm". You now have 2 note(s) saved.

Testing Habit Tracker:
A: Logged habit: "drink water". Total completions: 1.
A: Logged habit: "exercise". Total completions: 1.
A: Logged habit: "drink water". Total completions: 2.


## Summary

### What We Learned:
- Tools are functions AI can call
- Tool registry manages available tools
- ReAct pattern: Think → Act → Observe
- Agents can use tools autonomously
- Digital twin + tools = powerful assistant

### Next: Week 3.2 - Testing Your AI